## GSAT trend patterns

In [ ]:
# In[1]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
# %%
# define function
import src.SAT_function_Obs_Fingerprint as data_process
import src.Data_Preprocess as preprocess

In [ ]:
import src.slurm_cluster as scluster
client, scluster = scluster.init_dask_slurm_cluster()

In [ ]:
models = ['CanESM5', 'CESM2', 'IPSL_CM6A', 'EC_Earth3', 'ACCESS', 'MPI_ESM', 'MIROC6']
from pathlib import Path
base_dir = Path("/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3")

ICV_ds = {}  # dict: model -> Dataset

for m in models:
    dir_ICV_input = base_dir / m / "SMILE_internal"
    # adjust filename pattern if needed
    fn = dir_ICV_input / f"{m}_SMILE_noise_trend_std_sliding_1950_2022.nc"
    
    print(f"Opening {fn}")
    ICV_ds[m] = xr.open_dataset(fn)

In [ ]:
ICV_ds["ACCESS"]

In [ ]:
ICV_ds["CESM2"]

In [ ]:
# calculate the mean of SMILE std pattern
# MMEM trend can be calculated by averaging the trend from all models
ICV_trend_da = xr.concat([ICV_ds['CanESM5'].noise_trend_std,ICV_ds['IPSL_CM6A'].noise_trend_std,ICV_ds['CESM2'].noise_trend_std,
                         ICV_ds['EC_Earth3'].noise_trend_std,ICV_ds['ACCESS'].noise_trend_std,
                         ICV_ds['MPI_ESM'].noise_trend_std,ICV_ds['MIROC6'].noise_trend_std], dim='model', coords='minimal')
MMEM_ICV_trend_da = ICV_trend_da.mean(dim='model')

In [ ]:
ICV_trend_da

In [ ]:
# output the ensemble mean trend
import os
dir_out = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/MMLE/SMILE_internal/'
os.makedirs(dir_out, exist_ok=True)
MMEM_ICV_trend_da.to_dataset(name='trend').to_netcdf(dir_out + 'MMLE_internal_trend_std_1950-2022_sliding.nc')

In [ ]:
client.close()
scluster.close()